# 04 — Auditoría inicial de sospecha de infección

Explora fuentes de antimicrobianos y cultivos y prueba el emparejamiento temporal. **No genera todavía la etiqueta definitiva**: la lista farmacológica, vías sistémicas, evidencia de administración y especímenes deben revisarse y congelarse. Los recuentos son diagnósticos del demo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT
_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from mimic_sepsis.infection import pair_antibiotics_and_cultures
DATA_DIR = PROJECT_ROOT / 'data' / 'mimic-iv-demo' / '2.2' / 'hosp'

In [ ]:
prescriptions = pd.read_csv(DATA_DIR / 'prescriptions.csv.gz', low_memory=False)
emar = pd.read_csv(DATA_DIR / 'emar.csv.gz', low_memory=False)
microbiology = pd.read_csv(DATA_DIR / 'microbiologyevents.csv.gz', low_memory=False)
print(f'{len(prescriptions)} prescripciones, {len(emar)} eventos EMAR, {len(microbiology)} filas microbiológicas')

## Inventario farmacológico candidato

El patrón amplio siguiente sirve solo para encontrar nombres que revisar; incluye falsos positivos y no clasifica vías.

In [ ]:
candidate_pattern = (
    r'cef|cillin|penem|floxacin|mycin|cycline|azole|bactrim|trimeth|sulfameth|'
    r'metronidazole|linezolid|daptomycin|aztreonam|tigecycline'
)
drug_inventory = (
    prescriptions.loc[prescriptions['drug'].fillna('').str.contains(candidate_pattern, case=False, regex=True)]
    .groupby(['drug', 'route'], dropna=False).size().reset_index(name='prescriptions')
    .sort_values(['drug', 'route'])
)
drug_inventory

## Inventario y deduplicación de cultivos

Una solicitud puede producir varias filas por prueba, organismo o antibiograma. Se colapsa por `micro_specimen_id` antes de cualquier emparejamiento.

In [ ]:
specimen_inventory = (
    microbiology.groupby('spec_type_desc', dropna=False)['micro_specimen_id']
    .nunique().sort_values(ascending=False).rename('specimens').reset_index()
)

specimen_inventory.head(20)

In [ ]:
cultures = (
    microbiology.assign(culture_time=lambda x: pd.to_datetime(x['charttime']).fillna(pd.to_datetime(x['chartdate'])))
    .sort_values(['subject_id', 'hadm_id', 'micro_specimen_id', 'culture_time'])
    .drop_duplicates(['subject_id', 'hadm_id', 'micro_specimen_id'])
    .rename(columns={'micro_specimen_id': 'culture_id'})
)
blood_cultures = cultures[cultures['spec_type_desc'].fillna('').str.contains('BLOOD', case=False)].copy()
print(f'{len(cultures)} especímenes únicos; {len(blood_cultures)} clasificados como sangre')

## Prueba exploratoria del algoritmo temporal

Para probar el código se usa el subconjunto de prescripciones cuyo nombre coincide con el inventario candidato. Esto **no equivale** a una whitelist clínica y no debe emplearse como outcome.

In [ ]:
candidate_prescriptions = prescriptions.loc[
    prescriptions['drug'].fillna('').str.contains(candidate_pattern, case=False, regex=True)
] .copy()
candidate_prescriptions['antibiotic_time'] = pd.to_datetime(candidate_prescriptions['starttime'], errors='coerce')
candidate_prescriptions['antibiotic_id'] = candidate_prescriptions['pharmacy_id']
antibiotics = candidate_prescriptions[['subject_id','hadm_id','antibiotic_id','antibiotic_time']]
culture_events = cultures[['subject_id','hadm_id','culture_id','culture_time']]
candidate_pairs = pair_antibiotics_and_cultures(antibiotics, culture_events)
pd.DataFrame({
    'metric': ['candidate prescriptions', 'unique cultures', 'qualifying pairs', 'admissions with pairs'],
    'count': [len(antibiotics), len(culture_events), len(candidate_pairs), candidate_pairs['hadm_id'].nunique()],
})

In [ ]:
candidate_pairs.groupby('pair_direction').size().rename('pairs').reset_index()

## Decisiones obligatorias antes de etiquetar

1. Aprobar whitelist por nombre/código y rutas sistémicas.
2. Priorizar administración confirmada en EMAR frente a prescripción.
3. Definir episodios terapéuticos y dosis repetidas.
4. Elegir sangre como definición primaria y especímenes ampliados como sensibilidad.
5. Auditar profilaxis perioperatoria.
6. Mantener todos los pares candidatos hasta combinarlos con SOFA; no seleccionar retrospectivamente por conveniencia.